# 2.10 Actor-Critic：策略梯度 + 值函数

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.10_actor_critic.ipynb)


## 为什么要学

REINFORCE 用 MC 回报，方差大、要等回合结束。**Actor-Critic** 用一个学出来的值函数（critic，"评论家"）来替代 MC 回报，指导策略（actor，"演员"）更新：

- actor：$\pi(a|s;\boldsymbol\theta)$，负责选动作；
- critic：$v(s;\mathbf{w})$ 或 $q(s,a;\mathbf{w})$，负责评价动作好不好。

**这是 VLA+RL 最主流的结构**：

- RLinf 的 PPO：actor 是 VLA，critic 是加在 VLA 上的 value head；
- RLT：actor 是一个小 MLP（以 VLA 的参考动作为输入），critic 是 twin Q；
- RECAP：critic 是一个价值模型，用来计算优势，决定哪些数据值得模仿。

对应书中第 10 章。

---

## 1. QAC：最简单的 Actor-Critic

策略梯度 $\nabla J=\mathbb{E}[\nabla\ln\pi(A|S)\,q_\pi(S,A)]$ 中的 $q_\pi$，用 Sarsa 学出来的 $q(s,a;\mathbf{w})$ 代替：

- critic（Sarsa + 函数近似）：$\mathbf{w}\leftarrow\mathbf{w}+\alpha_w\big[r+\gamma q(s',a';\mathbf{w})-q(s,a;\mathbf{w})\big]\nabla_{\mathbf{w}}q(s,a;\mathbf{w})$
- actor：$\boldsymbol\theta\leftarrow\boldsymbol\theta+\alpha_\theta\nabla_{\boldsymbol\theta}\ln\pi(a|s;\boldsymbol\theta)\,q(s,a;\mathbf{w})$

## 2. A2C：优势 Actor-Critic

上一课说过，可以减去基线 $v(s)$ 降低方差，得到**优势函数**：

$$
A(s,a)=q(s,a)-v(s)
$$

而 $q(s,a)=\mathbb{E}[r+\gamma v(s')]$，所以**TD 误差就是优势的一个（有偏）估计**：

$$
\hat A_t = \delta_t = r_{t+1}+\gamma v(s_{t+1};\mathbf{w})-v(s_t;\mathbf{w})
$$

只需要一个状态值网络就够了。

- critic：最小化 $\big(r+\gamma v(s')-v(s)\big)^2$（TD 学习，目标不求梯度）；
- actor：最大化 $\ln\pi(a|s)\,\hat A$（$\hat A$ 不求梯度）。

## 3. GAE：在偏差与方差之间折中

一步 TD 误差偏差大（依赖不准的 $v$），MC 回报方差大。**n 步优势**是两者的中间：

$$
\hat A_t^{(n)}=r_{t+1}+\gamma r_{t+2}+\dots+\gamma^{n-1}r_{t+n}+\gamma^n v(s_{t+n})-v(s_t)=\sum_{k=0}^{n-1}\gamma^k\delta_{t+k}
$$

**广义优势估计（GAE）** 对所有 $n$ 做指数加权平均（权重 $(1-\lambda)\lambda^{n-1}$），化简后极其简洁：

$$
\boxed{\hat A_t^{GAE(\gamma,\lambda)}=\sum_{k=0}^{\infty}(\gamma\lambda)^k\delta_{t+k}}
\qquad\Longleftrightarrow\qquad
\hat A_t=\delta_t+\gamma\lambda\,\hat A_{t+1}
$$

- $\lambda=0$：一步 TD 误差（低方差、高偏差）；
- $\lambda=1$：MC 回报减基线（高方差、低偏差）；
- 常用 $\lambda=0.95$。

critic 的回归目标通常取 $\hat R_t=\hat A_t+v(s_t)$（"λ-回报"）。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.distributions import Categorical
from vla_ladder.envs import CartPole, SyncVecEnv
from vla_ladder.utils import moving_average, set_seed

### 练习 1：实现 GAE

处理"批量环境"的数据：形状都是 `(T, N)`，`T` 个时间步、`N` 个并行环境。`dones[t]=1` 表示第 `t` 步之后回合结束（之后的数据属于新回合，不能把它们的优势传过来）。

In [ ]:
def compute_gae(rewards, values, dones, last_values, gamma=0.99, lam=0.95):
    """
    rewards, values, dones: (T, N)；last_values: (N,) 是 rollout 结束后下一个状态的值 v(s_T)
    返回 advantages (T, N) 与 returns = advantages + values
    """
    T = rewards.shape[0]
    adv = torch.zeros_like(rewards)
    last_gae = torch.zeros_like(last_values)
    # TODO: 从 t = T-1 倒序到 0：
    #   next_value = last_values if t == T-1 else values[t+1]
    #   not_done = 1 - dones[t]
    #   delta = rewards[t] + gamma * next_value * not_done - values[t]
    #   last_gae = delta + gamma * lam * not_done * last_gae
    #   adv[t] = last_gae
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def compute_gae(rewards, values, dones, last_values, gamma=0.99, lam=0.95):
    """
    rewards, values, dones: (T, N)；last_values: (N,) 是 rollout 结束后下一个状态的值 v(s_T)
    返回 advantages (T, N) 与 returns = advantages + values
    """
    T = rewards.shape[0]
    adv = torch.zeros_like(rewards)
    last_gae = torch.zeros_like(last_values)
    for t in reversed(range(T)):
        next_value = last_values if t == T - 1 else values[t + 1]
        not_done = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * not_done - values[t]
        last_gae = delta + gamma * lam * not_done * last_gae
        adv[t] = last_gae
    return adv, adv + values

In [ ]:
# ✅ 自测
r = torch.tensor([[1.0], [1.0], [1.0]]); v = torch.tensor([[0.5], [0.4], [0.3]]); d = torch.zeros(3, 1)
adv, ret = compute_gae(r, v, d, torch.tensor([0.2]), gamma=0.9, lam=0.0)
assert torch.allclose(adv[:, 0], torch.tensor([1 + 0.9 * 0.4 - 0.5, 1 + 0.9 * 0.3 - 0.4, 1 + 0.9 * 0.2 - 0.3]))  # λ=0：一步 TD 误差
adv1, ret1 = compute_gae(r, v, d, torch.tensor([0.2]), gamma=0.9, lam=1.0)
assert torch.isclose(ret1[0, 0], torch.tensor(1 + 0.9 + 0.81 + 0.729 * 0.2))   # λ=1：MC 回报（末尾自举）
d2 = torch.tensor([[0.0], [1.0], [0.0]])                                          # 第 1 步后回合结束
adv2, _ = compute_gae(r, v, d2, torch.tensor([0.2]), gamma=0.9, lam=1.0)
assert torch.isclose(adv2[1, 0], torch.tensor(1 - 0.4))                           # 不会把下一回合的信息传过来
print("通过 ✅")

---

## 4. Off-policy Actor-Critic 与重要性采样

策略梯度是 on-policy 的：期望是对**当前策略** $\pi$ 采样的数据取的。如果数据来自另一个**行为策略** $\beta$（比如旧版本的策略、人类示教），可以用**重要性采样**修正：

$$
\mathbb{E}_{X\sim p}[f(X)]=\mathbb{E}_{X\sim\beta}\left[\frac{p(X)}{\beta(X)}f(X)\right]
$$

$\rho=\frac{p(x)}{\beta(x)}$ 叫重要性权重。于是 off-policy 策略梯度为

$$
\nabla J=\mathbb{E}_{A\sim\beta}\left[\frac{\pi(A|S)}{\beta(A|S)}\nabla\ln\pi(A|S)\,A_\pi(S,A)\right]
$$

**PPO 的核心就是这个比值** $r(\theta)=\frac{\pi_\theta(a|s)}{\pi_{\theta_{old}}(a|s)}$，再加上"裁剪"防止它太大（下一课）。

### 练习 2：重要性采样

$p=\mathcal{N}(1, 1)$，$\beta=\mathcal{N}(0, 1)$。只用来自 $\beta$ 的样本，估计 $\mathbb{E}_p[X^2]$（真实值 $=1+1=2$）。

In [ ]:
from torch.distributions import Normal

def importance_sampling_estimate(f, p, beta, n=200000):
    # TODO: x ~ beta；w = exp(p.log_prob(x) - beta.log_prob(x))；返回 (w * f(x)).mean()
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
from torch.distributions import Normal

def importance_sampling_estimate(f, p, beta, n=200000):
    x = beta.sample((n,))
    w = (p.log_prob(x) - beta.log_prob(x)).exp()      # 用 log 相减再 exp，数值更稳定
    return (w * f(x)).mean()

In [ ]:
# ✅ 自测
est = importance_sampling_estimate(lambda x: x ** 2, Normal(1.0, 1.0), Normal(0.0, 1.0))
print(f"重要性采样估计 E_p[X^2] = {est:.3f}（真实值 2）")
assert abs(est - 2) < 0.1
far = importance_sampling_estimate(lambda x: x ** 2, Normal(4.0, 1.0), Normal(0.0, 1.0), n=2000)
print(f"当两个分布差很远（p=N(4,1)）且样本少时：估计 = {far:.2f}（真实值 17）——几乎所有样本权重≈0，结果由极少数样本决定，方差极大！")
print("通过 ✅ —— 这就是为什么 PPO 要限制新旧策略不能差太多")

## 5. 确定性策略梯度（DPG）

连续动作时，可以让 actor 直接输出一个**确定性动作** $a=\mu(s;\boldsymbol\theta)$，此时策略梯度变成

$$
\nabla_{\boldsymbol\theta}J=\mathbb{E}_{S}\Big[\nabla_{\boldsymbol\theta}\mu(S)\,\nabla_a q(S,a)\big|_{a=\mu(S)}\Big]
$$

直觉：**沿着 critic 告诉你的"q 值上升最快的动作方向"移动动作**。代码中就是 `actor_loss = -critic(s, actor(s)).mean()`。DDPG、TD3 基于它；SAC 是它的随机版本；**RLT 的 actor 更新正是 `-Q(z, s, actor(...)) + β·BC 正则`**。DPG 天然是 off-policy 的（梯度里没有对动作分布的期望），所以可以用 replay buffer。

---

## 6. 实现 A2C（CartPole，8 个并行环境）

### 练习 3：A2C 的损失

In [ ]:
def a2c_loss(logits, values, actions, advantages, returns, vf_coef=0.5, ent_coef=0.01):
    """所有输入第 0 维都是 batch。返回 (总损失, 信息字典)"""
    # TODO:
    #   dist = Categorical(logits=logits)
    #   policy_loss = -(dist.log_prob(actions) * advantages.detach()).mean()
    #   value_loss = F.mse_loss(values, returns.detach())
    #   entropy = dist.entropy().mean()            # 熵奖励：鼓励探索，防止策略过早变成确定性
    #   loss = policy_loss + vf_coef * value_loss - ent_coef * entropy
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def a2c_loss(logits, values, actions, advantages, returns, vf_coef=0.5, ent_coef=0.01):
    """所有输入第 0 维都是 batch。返回 (总损失, 信息字典)"""
    dist = Categorical(logits=logits)
    policy_loss = -(dist.log_prob(actions) * advantages.detach()).mean()
    value_loss = F.mse_loss(values, returns.detach())
    entropy = dist.entropy().mean()
    loss = policy_loss + vf_coef * value_loss - ent_coef * entropy
    return loss, {"policy_loss": policy_loss.item(), "value_loss": value_loss.item(), "entropy": entropy.item()}

In [ ]:
class ActorCritic(nn.Module):
    """actor 与 critic 各自一个小 MLP（也可以共享底层特征，像 RLinf 给 VLA 加 value head 那样）"""

    def __init__(self, obs_dim=4, n_actions=2, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, n_actions))
        self.critic = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, obs):
        return self.actor(obs), self.critic(obs).squeeze(-1)


def train_a2c(total_steps=100_000, n_envs=8, n_steps=16, gamma=0.99, lam=0.95, lr=7e-4, seed=0):
    set_seed(seed)
    venv = SyncVecEnv([CartPole for _ in range(n_envs)])
    model = ActorCritic()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    obs = torch.as_tensor(venv.reset(seed=seed))
    returns_log, steps_log, step = [], [], 0
    while step < total_steps:
        buf = {k: [] for k in ["obs", "actions", "rewards", "dones", "values"]}
        for t in range(n_steps):                                   # 1) 收集 n_steps × n_envs 的数据
            with torch.no_grad():
                logits, value = model(obs)
            action = Categorical(logits=logits).sample()
            next_obs, r, term, trunc, info = venv.step(action.numpy())
            r = torch.as_tensor(r)
            # 被截断（truncated）的回合：把 γ v(最后观测) 加进奖励，相当于"自举"它的未来
            for i in np.nonzero(trunc & ~term)[0]:
                with torch.no_grad():
                    r[i] += gamma * model(torch.as_tensor(info["final_obs"][i]))[1]
            for k, v in zip(buf, [obs, action, r, torch.as_tensor(term | trunc, dtype=torch.float32), value]):
                buf[k].append(v)
            obs = torch.as_tensor(next_obs); step += n_envs
            returns_log += info["episode_returns"]; steps_log += [step] * len(info["episode_returns"])
        with torch.no_grad():
            last_value = model(obs)[1]
        data = {k: torch.stack(v) for k, v in buf.items()}          # 每项 (T, N, ...)
        adv, ret = compute_gae(data["rewards"], data["values"], data["dones"], last_value, gamma, lam)  # 2) 估计优势
        logits, values = model(data["obs"].reshape(-1, 4))          # 3) 一次梯度更新
        loss, info_l = a2c_loss(logits, values, data["actions"].reshape(-1), adv.reshape(-1), ret.reshape(-1))
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 0.5); opt.step()
    return model, returns_log, steps_log

model, rets, steps = train_a2c()
plt.figure(figsize=(6, 3.5)); plt.plot(steps, rets, ".", alpha=0.2); plt.plot(steps[19:], moving_average(rets, 20))
plt.xlabel("env steps"); plt.ylabel("episode return"); plt.title("A2C on CartPole"); plt.show()
print("最后 20 个回合平均回报:", np.mean(rets[-20:]))
assert np.mean(rets[-20:]) > 150

---

## 小结

| 算法 | actor 的"权重" | critic 学什么 | on/off |
|---|---|---|---|
| REINFORCE | MC 回报 $g_t$ | 无 | on |
| QAC | $q(s,a)$ | Sarsa | on |
| A2C | TD 误差 / GAE 优势 | $v(s)$ | on |
| Off-policy AC | $\rho\cdot\hat A$ | $v(s)$ | off |
| DPG / DDPG / TD3 | $\nabla_a q(s,a)$ | $q(s,a)$ | off |

- GAE：$\hat A_t=\delta_t+\gamma\lambda\hat A_{t+1}$，用 $\lambda$ 调节偏差-方差。
- 重要性采样让旧数据可以用于新策略，但两个分布差太多时方差爆炸。
- 熵奖励防止策略过早收敛。

下一课 PPO：A2C 每批数据只用一次就丢掉，PPO 用重要性比值 + 裁剪，让**同一批数据可以安全地多更新几次**。